# Set up

In [ ]:
%run fabric-domain-management-util

In [ ]:
%run fabric-workspace-delete-util

# Tenant level clean up

This Notebook expects someone to be reading through the Notebook as it runs - it's not to be run automatically

In [ ]:
# Check if there's any Domains with no Workspaces assigned to them which could be deleted 
empty_domains = check_empty_domains()
display(empty_domains)

**If there are domains to go delete, go run the Domain Management script, which will invoke `delete_domains(...)` for you**

# Asset level clean up

In [ ]:
with labs.service_principal_authentication(
        key_vault_uri = key_vault_uri,
        key_vault_tenant_id = 'tenant-fabric-sp',
        key_vault_client_id = 'fabric-admin-sp-app-id', 
        key_vault_client_secret = admin_sp_client_secret_name[1]
):
    orphaned_workspaces = labs.admin.list_orphaned_workspaces()
    # keep only those not already deleted, and exclude personal workspaces as they're cleaned up autoatically anyway
    orphaned_workspaces = orphaned_workspaces[(orphaned_workspaces['State'] != 'Deleted') & (orphaned_workspaces['Type'] != 'PersonalGroup')]
    content_published_to_web = labs.admin.list_widely_shared_artifacts('PublishedToWeb')
    content_using_global_links = labs.admin.list_widely_shared_artifacts('LinksSharedToWholeOrganization')
    overshared_reports = content_using_global_links[content_using_global_links['Artifact Type'] == 'Report']

## What Workspaces have no owners

### Workspaces with no users at all!!
If the content in here is stale, delete the workspaces

In [ ]:
# workspaces w/ no users at all (is na)
abandoned_workspaces = orphaned_workspaces[pd.isna(orphaned_workspaces['Users'])]
print("abandoned workspaces: ", abandoned_workspaces.shape[0])
display(abandoned_workspaces)

### Workspaces with no admins, but some other users
Contact these people & identify if there is someone who should own these, or if they can be transferred to another env

In [ ]:
# workspaces w/ no admins but some users (not na)
orphaned_workspaces_some_users = orphaned_workspaces[pd.notna(orphaned_workspaces['Users'])]
display(orphaned_workspaces_some_users)

### Inventory & Clean up scripts

In [ ]:
def check_lost_content(input_workspace_df):
    """
    This function processes an input DataFrame containing workspace information, retrieves unowned content (reports, semantic models, and dataflows) 
    from each workspace, and returns the consolidated data for each content type. If the input DataFrame is empty, the function returns `None`.

    Parameters
    ----------
    input_workspace_df : pandas.DataFrame
        A DataFrame containing workspace information. It must include a column named 'Workspace Id' that identifies each workspace.

    Returns
    -------
    tuple of pandas.DataFrame or None
        If the input DataFrame is not empty, the function returns a tuple containing three DataFrames:
        - lost_workspace_reports : pandas.DataFrame
            A DataFrame containing information about unowned reports, merged with the input workspace information.
        - lost_workspace_models : pandas.DataFrame
            A DataFrame containing information about unowned semantic models, merged with the input workspace information.
        - lost_workspace_dfs : pandas.DataFrame
            A DataFrame containing information about unowned dataflows, merged with the input workspace information.
        
        If the input DataFrame is empty, the function returns `None`.

    Notes
    -----
    - This function uses a service principal for authentication via the `labs.service_principal_authentication` method.
    - The function assumes the presence of external variables such as `key_vault_uri`, `key_vault_tenant_id`, `key_vault_client_id`, 
      and `admin_sp_client_secret_name` for authentication purposes.
    - The `labs.admin.list_items` method is used to fetch the unowned content for each workspace by type ('Report', 'SemanticModel', 'Dataflow').

    """

    # if df is empty, do nothing
    if input_workspace_df.size == 0:
        return(None)
    # otherwise kick off processes
    else: 
        # 1. Initialize an empty list to store the dataframes
        temp_lost_reports = []
        temp_lost_models = []
        temp_lost_dfs = []
        # 2. Loop through your data source, generate dataframes, and append them to the lis
        for index, item_row in input_workspace_df.iterrows():
            # fetch the diff types of item
            with labs.service_principal_authentication(
                key_vault_uri = key_vault_uri,
                key_vault_tenant_id = 'tenant-fabric-sp',
                key_vault_client_id = 'fabric-admin-sp-app-id', 
                key_vault_client_secret = admin_sp_client_secret_name[1]
            ):
                temp_workspace_report = labs.admin.list_items(workspace = item_row['Workspace Id'], type = 'Report')
                temp_workspace_model = labs.admin.list_items(workspace = item_row['Workspace Id'], type = 'SemanticModel')
                temp_workspace_df = labs.admin.list_items(workspace = item_row['Workspace Id'], type = 'Dataflow')
            # merge all the type dfs
            temp_lost_reports.append(temp_workspace_report)
            temp_lost_models.append(temp_workspace_model)
            temp_lost_dfs.append(temp_workspace_df)

        # convert list of DFS to 1x DF per item type
        lost_workspace_reports = pd.concat(temp_lost_reports, ignore_index=True)
        lost_workspace_models = pd.concat(temp_lost_models, ignore_index=True)
        lost_workspace_dfs = pd.concat(temp_lost_dfs, ignore_index=True)

        # get workspace info
        lost_workspace_reports = pd.merge(lost_workspace_reports, input_workspace_df, 'inner', left_on='Workspace Id', right_on= 'Workspace Id')
        lost_workspace_models = pd.merge(lost_workspace_models, input_workspace_df, 'inner', left_on='Workspace Id', right_on= 'Workspace Id')
        lost_workspace_dfs = pd.merge(lost_workspace_dfs, input_workspace_df, 'inner', left_on='Workspace Id', right_on= 'Workspace Id')

    return (lost_workspace_reports, lost_workspace_models, lost_workspace_dfs)

In [ ]:
abandoned_workspace_content = check_lost_content(abandoned_workspaces)
orphaned_workspace_content  = check_lost_content(orphaned_workspaces_some_users)

In [ ]:
if abandoned_workspace_content != None:
    print("Content with no users")
    display(abandoned_workspace_content[0])
    display(abandoned_workspace_content[1])
    display(abandoned_workspace_content[2])

if orphaned_workspace_content != None:
    print ("Content with no Admin, but some users left over")
    display(orphaned_workspace_content[0])
    display(orphaned_workspace_content[1])
    display(orphaned_workspace_content[2])

#### DECISION POINT: DELETE THESE ABANDONED WORKSPACES?

In [ ]:
# if all items are no longer required - proceed with deletion of workspace (and all items inside)
# these asset bundles have no owner, no other users (non workspace admin), and have not been modified recently

## REQUIRES YOU TO MAKE A DECISION

# if df is empty, do nothing
if abandoned_workspaces.size == 0:
    print('Nothing to do')
# otherwise kick off processes
else: 
    for index, row in abandoned_workspaces.iterrows():
        delete_workspace_access(row['Workspace Id'])

#### DECISION POINT - DELETE THESE ORPHANED WORKSPACES?
You should probably go contact those users and ask them what they want to do with the content

In [ ]:
# if all items are no longer required - proceed with deletion of workspace (and all items inside)
# these asset bundles have no owner, no other users (non workspace admin), and have not been modified recently

## REQUIRES YOU TO MAKE A DECISION

# if df is empty, do nothing
if abandoned_workspaces.size == 0:
    print('Nothing to do')
# otherwise kick off processes
else: 
    for index, row in orphaned_workspaces_some_users.iterrows():
        delete_workspace_access(row['Workspace Id'])

## What is public

In [ ]:
count_web_links = content_published_to_web.shape[0]
print("There are ", count_web_links, " reports exposed on the public internet")
display(content_published_to_web)

## What's been shared with the whole org

In [ ]:
display(overshared_reports)

In [ ]:
# data prep for plotting worst offenders
res_count_overshare = pd.DataFrame(overshared_reports.groupby('Sharer Email Address')['Artifact Id'].count())
res_count_overshare = res_count_overshare.reset_index()
res_count_overshare.columns = ['Sharer', "Share Count"]
res_count_overshare = res_count_overshare.sort_values('Share Count')

In [ ]:
def plot_high_sharers(input_df, thresh):
    input_df = input_df[input_df['Share Count'] > thresh]
    plt.barh(
        input_df['Sharer'],
        input_df['Share Count']
    )
    plt.show()

In [ ]:
plot_high_sharers(res_count_overshare, 15)

In [ ]:
res_count_sharetype = pd.DataFrame(content_using_global_links.groupby('Artifact Type')['Artifact Id'].count())
res_count_sharetype